In [10]:
## Day 3 — Python for Data Cleaning (Basic)

# Objective:
# Clean and enrich raw ride-sharing data so it becomes analytics-ready
# for SQL analysis and dashboards.

# Why this step exists:
# Operational data is often incomplete, inconsistent, and not directly usable.
# This step ensures data quality and creates derived features required for
# demand analysis, cancellation analysis, and fraud detection.

In [45]:
# Inspect column names before writing transformations
# This prevents incorrect assumptions about the data

rides.columns

Index(['ride_id', 'user_id', 'driver_id', 'requested_at', 'ride_status',
       'pickup_lat', 'pickup_lng', 'drop_lat', 'drop_lng', 'distance_km',
       'duration_sec', 'fare', 'hour', 'day', '15_min_bucket'],
      dtype='object')

In [46]:
import pandas as pd
import numpy as np

In [48]:
# Load raw CSV files into pandas DataFrames
# CSV files usually come directly from data pipelines or data dumps

rides = pd.read_csv("..\\..\\Ride\\rides.csv")
users = pd.read_csv("..\\..\\Ride\\users.csv")
drivers = pd.read_csv("..\\..\\Ride\\drivers.csv")


In [51]:
# Define critical location columns
# Location data is required for distance calculation and spatial analysis

location_cols = ["pickup_lat", "pickup_lng", "drop_lat", "drop_lng"]

# Remove rows where any location value is missing
rides = rides.dropna(subset=location_cols)


In [52]:
# Convert requested_at from string to datetime
# Required for time-based analysis

rides["requested_at"] = pd.to_datetime(rides["requested_at"])


In [53]:
# Convert requested_at from string to datetime
rides["requested_at"] = pd.to_datetime(rides["requested_at"])

# Extract hour
rides["hour"] = rides["requested_at"].dt.hour

# Extract day name
rides["day"] = rides["requested_at"].dt.day_name()

# Create 15-minute bucket
rides["15_min_bucket"] = rides["requested_at"].dt.floor("15min")


In [54]:
# Haversine formula calculates real-world distance
# between two latitude-longitude points on Earth

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers
    
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2)**2 + \
        np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    
    return 2 * R * np.arcsin(np.sqrt(a))

# Vectorized distance calculation (NO loops)
rides["distance_km"] = haversine(
    rides["pickup_lat"], rides["pickup_lng"],
    rides["drop_lat"], rides["drop_lng"]
)


In [55]:
# Save cleaned and enriched dataset
# This dataset will be loaded into MySQL for analysis

rides.to_csv("rides_cleaned.csv", index=False)
